# Vector Store
## FAISS
Facebook AI Similarity Search (FAISS) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to that possibly do not fit in RAM. It also conatins supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("examples/speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


d:\2026-courses\agenticai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs

[Document(metadata={'source': 'examples/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'examples/speech.txt'}, page_content='â€¦\n\nIt will be all the easier

In [3]:
embedding=OllamaEmbeddings(model="gemma:2b")
db=FAISS.from_documents(docs,embedding)
db

C:\Users\pc\AppData\Local\Temp\ipykernel_9512\2688668785.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding=OllamaEmbeddings(model="gemma:2b")


In [4]:
### Querying
query="How does the speaker describe the desired outcome of the war?"
docs= db.similarity_search(query)

In [6]:
docs[0].page_content

'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [7]:
retriever= db.as_retriever()
retriever.invoke(query)
docs[0].page_content

'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

## Note:
The reason for converting a vector store to a retriever using db.as_retriever() is to provide a standardized, flexible interface for document retrieval. Here are the key benefits:

### Main Reasons:
- Standardized Interface: The retriever provides a consistent .invoke() or .get_relevant_documents() method that works across different retrieval strategies, making it easier to swap or chain different retrievers.

- Integration with LangChain Chains: Retrievers are designed to work seamlessly with LangChain's chains (like RetrievalQA, ConversationalRetrievalChain). These chains expect a retriever object, not a raw vector store.

- Flexible Search Parameters: When converting to a retriever, you can configure:
    - search_type: Choose between "similarity", "mmr" (Maximum Marginal Relevance), or "similarity_score_threshold"
    - search_kwargs: Set parameters like k (number of documents), score_threshold, etc.
- Abstraction: It abstracts away the underlying vector store implementation. Your code can work with any retriever (vector store-based, web search, hybrid, etc.) without changing the calling code.

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [14]:
docs_and_score=db.similarity_search_with_score(query)
print(docs_and_score)

[(Document(id='1eca2dc9-648a-4de5-9187-b24b6a686d37', metadata={'source': 'examples/speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'), np.float32(2971.4722)), (Document(id='2c54f87e-6edd-485c-bad5-c45b1c4625a2', metadata={'source': 'examples/speech.txt'}, page_content='We have borne with their present government through all these bitter mon

In [15]:
embedding_vector=embedding.embed_query(query)
embedding_vector

[0.2651284635066986,
 -2.164094924926758,
 0.24361425638198853,
 0.9848988652229309,
 0.6762158870697021,
 0.5160541534423828,
 -0.8752997517585754,
 0.12181451171636581,
 -0.11031506210565567,
 -0.7873508334159851,
 1.0992497205734253,
 -0.028023259714245796,
 -1.1107875108718872,
 1.1126807928085327,
 0.061422593891620636,
 -0.9937295913696289,
 3.1106934547424316,
 1.7364500761032104,
 1.1433651447296143,
 0.7742565274238586,
 0.33120599389076233,
 -0.30817487835884094,
 0.3038901090621948,
 1.3740729093551636,
 0.19731423258781433,
 -0.41631796956062317,
 -1.3816300630569458,
 -1.2693721055984497,
 -0.5324482917785645,
 -2.1077353954315186,
 -0.2801658809185028,
 -1.4480514526367188,
 1.2137185335159302,
 -0.9418930411338806,
 -0.39754927158355713,
 -0.25710228085517883,
 1.9100379943847656,
 0.5486786961555481,
 0.11091141402721405,
 -0.5293307900428772,
 0.37347450852394104,
 0.5500903129577637,
 0.955491840839386,
 -1.3187646865844727,
 -1.39031183719635,
 0.6351297497749329,
 0

In [16]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='1eca2dc9-648a-4de5-9187-b24b6a686d37', metadata={'source': 'examples/speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='2c54f87e-6edd-485c-bad5-c45b1c4625a2', metadata={'source': 'examples/speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that frien

In [17]:
### Saving and Loading
db.save_local("faiss_index")


In [18]:
new_db=FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [19]:
docs

[Document(id='1eca2dc9-648a-4de5-9187-b24b6a686d37', metadata={'source': 'examples/speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='2c54f87e-6edd-485c-bad5-c45b1c4625a2', metadata={'source': 'examples/speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that frien

## Chroma 

In [21]:
## building a sample vectordb
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
loader = TextLoader("examples/speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'examples/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and 

In [24]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

In [25]:
embedding=OllamaEmbeddings(model="gemma:2b")
vectordb=Chroma.from_documents(documents=splits,embedding=embedding)
vectordb

In [26]:
## query it
query = "What does the speaker believe is the main reason the United States should enter the war?"
docs = vectordb.similarity_search(query)
docs[0].page_content

'democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

In [27]:
## saving to disk
vectordb=Chroma.from_documents(documents=splits,embedding=embedding,persist_directory="./chroma_db")

In [28]:
## Load 
db2= Chroma(persist_directory="./chroma_db",embedding_function=embedding)


In [29]:
## similarity Search With Score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='38b70843-eaaa-4bf2-bd00-b96658031979', metadata={'source': 'examples/speech.txt'}, page_content='democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  3495.435302734375),
 (Document(id='9fd271b7-8cf2-4d57-a3f0-346d2725aa4f', metadata={'source': 'examples/speech.txt'}, page_content='government in the hour of test. They are, most of them, as true and loyal Americans as if they had never known any other fealty or allegiance. They will be prompt to stand with us in rebuking and restraining the few who may be of a different mind and purpose. If there should be disloyalty, it will be dealt with with a firm hand of stern repression; but, if it lifts its head at all, it will lift it only here and there and without countenance exce

In [30]:
### Retriever option
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'